# Week 5: Ensemble Learning & Decision Trees
## Notebook 06: CatBoost, Ordered Boosting & Oblivious Trees from Scratch

### 1. Conceptual Intuition & Real-World Use Case
While XGBoost and LightGBM dominate tabular data competitions, they suffer from two fundamental weaknesses:
1. **Prediction Shift (Target Leakage):** In standard gradient boosting, the gradients used to grow trees are calculated on the same data points evaluated by the tree, causing optimistic gradient bias and overfitting.
2. **Categorical Handling:** One-Hot Encoding blows up dimensionality; naive target encoding causes target leakage.

In 2018, Yandex released **CatBoost (Categorical Boosting)**:
- **Ordered Boosting:** Simulates sequential time-series arrivals on random permutations of data to calculate completely unbiased gradients.
- **Symmetric (Oblivious) Trees:** Trees split using the exact same feature condition across all nodes at each depth level, resulting in ultra-fast CPU inference via binary bitwise lookups.

---

### 2. Mathematical Foundations: Ordered Target Statistics
For observation $x_k$ under random permutation $\sigma = (x_{(1)}, x_{(2)}, \dots, x_{(n)})$:

$$\hat{x}_{k, c} = \frac{\sum_{j=1}^{k-1} \mathbb{I}(x_{(j), c} = x_{(k), c}) \cdot y_{(j)} + a \cdot p}{\sum_{j=1}^{k-1} \mathbb{I}(x_{(j), c} = x_{(k), c}) + a}$$

Notice: **Only observations strictly preceding $x_{(k)}$ in the permutation are used to compute its target statistic!**
Future observations are never leaked, eliminating prediction shift.

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Create Dataset with Categorical Features
np.random.seed(42)
N = 8

categories = ["Tech", "Finance", "Tech", "Healthcare", "Finance", "Tech", "Healthcare", "Finance"]
targets = [1, 0, 1, 0, 1, 1, 0, 0]

df = pd.DataFrame({"category": categories, "target": targets})
print("Raw Dataset:\n", df)

# Step 2: Implement CatBoost Ordered Target Encoding
def catboost_ordered_target_encoding(df, cat_col, target_col, prior=0.5, a=1.0):
    # Step A: Generate random permutation of indices
    permuted_indices = np.random.permutation(len(df))
    
    encoded_values = np.zeros(len(df))
    running_counts = {}
    running_sums = {}
    
    for idx in permuted_indices:
        cat_val = df.loc[idx, cat_col]
        
        # Current category historical stats before this index
        cat_count = running_counts.get(cat_val, 0)
        cat_sum = running_sums.get(cat_val, 0.0)
        
        # Compute Ordered Target Statistic: (sum + a * prior) / (count + a)
        encoded_val = (cat_sum + a * prior) / (cat_count + a)
        encoded_values[idx] = encoded_val
        
        # Update running stats WITH current target (available only for future samples)
        running_counts[cat_val] = cat_count + 1
        running_sums[cat_val] = cat_sum + df.loc[idx, target_col]
        
    return encoded_values

df["ordered_encoded"] = catboost_ordered_target_encoding(df, "category", "target")
print("\n--- Dataset with CatBoost Ordered Target Encoding (Unbiased) ---")
print(df)

# Step 3: Oblivious (Symmetric) Tree Principle
# In an oblivious tree, all nodes at depth d share the IDENTICAL splitting criterion
print("\n--- Oblivious Tree Structure vs Standard Asymmetric Tree ---")
print("Depth 1: Feature 0 > 2.5 ? (Applies to all leaves)")
print("Depth 2: Feature 1 <= 0.8 ? (Applies to all 2 leaves simultaneously -> 4 terminal leaves)")
print("Benefit: The leaf index can be evaluated with a simple binary bitmask: (f0 > 2.5) << 1 | (f1 <= 0.8)")

### 3. Step-by-Step Code Walkthrough
1. **Permutation Simulation:** Simulates an online data arrival stream.
2. **Prior Dominance:** The very first time "Finance" appears, it has 0 historical samples, so it cleanly defaults to the prior ($0.50$).
3. **Unbiased Gradients:** Each sample's feature representation is constructed strictly from historical context, breaking the circular dependency between target and features.

---

### 4. Industry Applications & Trade-offs
- **Production Latency:** Oblivious trees compile directly into CPU SIMD vector instructions, evaluating tens of thousands of predictions per millisecond.
- **Out-of-the-Box Tuning:** CatBoost generally achieves top performance without requiring extensive hyperparameter searches.

---

### 5. Top Interview Questions & Answers
**Q1: What is an Oblivious Tree and why does CatBoost use it?**
> **Answer:** An oblivious (symmetric) tree uses the exact same feature and split threshold across all nodes at a given depth level. This acts as strong regularization (preventing overfitting) and allows execution as a simple table lookup with bit shifts, making inference 10x faster than asymmetric trees.

**Q2: What is "Prediction Shift" in standard gradient boosting?**
> **Answer:** In standard GBDT, the gradient $g(x_i, y_i)$ at step $t$ is computed using model $F_{t-1}(x_i)$ trained on the dataset containing $x_i$. This causes the distribution of gradients on training data to shift relative to unseen test data, leading to overfitting. Ordered boosting eliminates this by using models trained strictly on random prefixes.